# Specification

Core do problema (2 frases técnicas)

Projetar uma estrutura de dados que suporte acesso e atualização em O(1), mantendo uma política de remoção baseada em Least Recently Used (LRU).
Sempre que a capacidade for excedida, devemos remover o item menos recentemente acessado (via get ou put).

## Edge Cases & Traps
Capacidade = 1 → sempre remove o anterior
Chave já existe no put → não duplicar, atualizar e mover
get em chave inexistente → retornar -1
Não mover node no get → quebra o LRU silenciosamente
Remoção incorreta na lista → pointers quebram → bug difícil (clássico em entrevista)

# Plan

🧠 Heurística

HashMap + Doubly Linked List

HashMap → acesso O(1)
Lista duplamente encadeada → manter ordem de uso em O(1)

👉 Essa combinação vence qualquer alternativa:

Array ❌ (remoção O(n))
Heap ❌ (não mantém ordem de uso corretamente)

## Desenho

HEAD <-> ... <-> ... <-> TAIL
 ↑                       ↑
MRU                    LRU

MRU -> most recent used
LRU -> least recent used

HEAD → mais recente
TAIL → menos recente

## Operações-chave

get(key)

    Se não existe → retorna -1

    Se existe:
        move para HEAD
        retorna valor   


put(key, value)
    Se key existe:
        atualiza valor
        move para HEAD

    Se key não existe:
        cria node
        insere no HEAD

    Se passou capacity:
        remove TAIL.prev (LRU)

Visual Trace (Passo a passo)

## Exemplo:

Input: ["LRUCache", "put", "put", "get", "put", "get", "put", "get", "get", "get"]
Output: [[2], [1, 1], [2, 2], [1], [3, 3], [2], [4, 4], [1], [3], [4]]


capacity = 2
Estado inicial

HEAD <-> TAIL
put(1,1)

HEAD <-> 1 <-> TAIL
put(2,2)

HEAD <-> 2 <-> 1 <-> TAIL
get(1)

move 1 para frente

HEAD <-> 1 <-> 2 <-> TAIL
put(3,3)

remove LRU (2)

HEAD <-> 3 <-> 1 <-> TAIL

A lista representa tempo de uso, não ordem de inserção.

In [ ]:
from typing import Optional


class Node:
    def __init__(self, key: int, value: int):
        self.key: int = key
        self.value: int = value
        self.prev: Optional['Node'] = None
        self.next: Optional['Node'] = None


class LRUCache:

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key -> Node  # O(1)

        # Dummy nodes (evita edge cases)
        self.head = Node(0, 0)
        self.tail = Node(0, 0)

        self.head.next = self.tail
        self.tail.prev = self.head

    # 🔧 Remove node da lista
    def _remove(self, node: Node) -> None:
        prev_node = node.prev
        next_node = node.next

        prev_node.next = next_node
        next_node.prev = prev_node
        # O(1)

    # 🔧 Insere logo após HEAD (mais recente)
    def _insert(self, node: Node) -> None:
        next_node = self.head.next

        self.head.next = node
        node.prev = self.head

        node.next = next_node
        next_node.prev = node
        # O(1)

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1  # O(1)

        node = self.cache[key]

        # Move para frente (mais recente)
        self._remove(node)   # O(1)
        self._insert(node)   # O(1)

        return node.value    # O(1)

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            # remove node antigo
            self._remove(self.cache[key])  # O(1)

        new_node = Node(key, value)
        self.cache[key] = new_node

        self._insert(new_node)  # O(1)

        if len(self.cache) > self.capacity:
            # remove LRU (último antes do tail)
            lru = self.tail.prev
            self._remove(lru)          # O(1)
            del self.cache[lru.key]   # O(1)

## Test Cases

In [ ]:
# Example 1
print("Example 1: Capacity = 2")
lru = LRUCache(2)
lru.put(1, 1)
lru.put(2, 2)
print(f"get(1) = {lru.get(1)}")
lru.put(3, 3)  # Evict 2
print(f"get(2) = {lru.get(2)}")
lru.put(4, 4)  # Evict 1
print(f"get(1) = {lru.get(1)}")
print(f"get(3) = {lru.get(3)}")
print(f"get(4) = {lru.get(4)}")

print("\n" + "="*50 + "\n")

# Example 2
print("Example 2: Capacity = 1")
lru2 = LRUCache(1)
lru2.put(0, 0)
print(f"get(0) = {lru2.get(0)}")
lru2.put(1, 1)  # Evict 0
print(f"get(0) = {lru2.get(0)}")

print("\n" + "="*50 + "\n")

# Example 3: Multiple operations
print("Example 3: Complex operations")
lru3 = LRUCache(3)
lru3.put(1, "apple")
lru3.put(2, "banana")
lru3.put(3, "cherry")
print(f"get(1) = {lru3.get(1)}")
lru3.put(4, "date")  # Evict 2 (LRU)
print(f"get(2) = {lru3.get(2)}")
lru3.put(5, "elderberry")  # Evict 3 (LRU now)
print(f"get(3) = {lru3.get(3)}")
lru3.put(1, "apple_v2")  # Update 1
print(f"get(1) = {lru3.get(1)}")

## Pointer Manipulation Details

### Adding to end (before tail):
```
Before: ... ↔ tail.prev ↔ tail

After:  ... ↔ tail.prev ↔ node ↔ tail

Code:
  node.next = tail              # node → tail
  node.prev = tail.prev         # node ← tail.prev
  tail.prev.next = node         # tail.prev → node
  tail.prev = node              # tail ← node
```

### Removing node:
```
Before: A ↔ B ↔ C

After:  A ↔ C (B removed)

Code:
  B.prev.next = B.next     # A → C
  B.next.prev = B.prev     # C ← A
```

## Interview Tips

1. **Draw the structure**: Diagram the DLL with dummy nodes
2. **Test edge cases**: 
   - Capacity = 1
   - Update existing keys
   - Accessing after eviction
3. **Explain pointer logic**: Be clear about why dummy nodes help
4. **Time complexity**: Emphasize O(1) all operations
5. **Real-world context**: Mention caching, CPU cache eviction policies

## Alternatives Discussed in Interview
- **OrderedDict** (Python 3.7+): Maintains insertion order, simpler but less educational
- **Using just HashMap**: Can't achieve O(1) eviction
- **Using array + sorting**: Would be O(n log n) for eviction